# Krishi Drishti - P07: Dataset Audit and Clean Stratified Splits
This notebook downloads (or loads), audits, detects near-duplicates using perceptual hashing, and generates a leakage-free 70/15/15 stratified train/val/test split for Tomato and Potato disease classes.

### Step 1: Kaggle API Setup (3 Simple Steps)
1. Go to your Kaggle Account Settings -> API -> Click **Create New Token** to download `kaggle.json`.
2. Upload `kaggle.json` to Google Colab or place in `~/.kaggle/kaggle.json`.
3. Set file permissions: `chmod 600 ~/.kaggle/kaggle.json`.


In [ ]:
# 1. Verify Dataset Path and Target Classes
import os
import json
import hashlib
from collections import defaultdict
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CLASS_MAPPING = {
    "Potato___Early_blight": "potato_early_blight",
    "Potato___Late_blight": "potato_late_blight",
    "Potato___healthy": "potato_healthy",
    "Tomato___Bacterial_spot": "tomato_bacterial_spot",
    "Tomato___Early_blight": "tomato_early_blight",
    "Tomato___Late_blight": "tomato_late_blight",
    "Tomato___Leaf_Mold": "tomato_leaf_mold",
    "Tomato___Septoria_leaf_spot": "tomato_septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "tomato_spider_mites",
    "Tomato___Target_Spot": "tomato_target_spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "tomato_yellow_leaf_curl_virus",
    "Tomato___Tomato_mosaic_virus": "tomato_mosaic_virus",
    "Tomato___healthy": "tomato_healthy"
}

dataset_dir = "Dataset/plantvillage dataset/color"
print(f"Targeting {len(CLASS_MAPPING)} classes for Tomato and Potato.")


### 2. Audit Images: Resolution, Integrity, Exact Hashes, and Perceptual dHash


In [ ]:
def compute_md5(path):
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def compute_dhash(image, hash_size=8):
    resized = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    pixels = np.array(resized.getdata(), dtype=np.int32).reshape((hash_size, hash_size + 1))
    diff = pixels[:, 1:] > pixels[:, :-1]
    return "".join(["1" if b else "0" for b in diff.flatten()])

records = []
seen_md5 = set()
for folder, model_class in CLASS_MAPPING.items():
    folder_path = os.path.join(dataset_dir, folder)
    if not os.path.exists(folder_path):
        continue
    for fname in os.listdir(folder_path):
        if fname.startswith("."):
            continue
        fpath = os.path.join(folder_path, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                size = img.size
                dh = compute_dhash(img)
            f_md5 = compute_md5(fpath)
            if f_md5 in seen_md5:
                continue
            seen_md5.add(f_md5)
            records.append({
                "path": fpath,
                "label": model_class,
                "dhash": dh,
                "width": size[0],
                "height": size[1]
            })
        except Exception as e:
            print(f"Corrupt image {fpath}: {e}")

df = pd.DataFrame(records)
print(f"Total valid non-duplicate images: {len(df)}")
df["label"].value_counts()


### 3. Stratified 70 / 15 / 15 Train, Validation, and Test Split


In [ ]:
# Stratified split ensures exact class distribution without data leakage
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

train_df = train_df.copy(); train_df["split"] = "train"
val_df = val_df.copy(); val_df["split"] = "val"
test_df = test_df.copy(); test_df["split"] = "test"

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Verification assertions
assert (split_df["split"] == "train").sum() + (split_df["split"] == "val").sum() + (split_df["split"] == "test").sum() == len(split_df)
assert len(set(train_df["path"]).intersection(set(test_df["path"]))) == 0
assert len(set(val_df["path"]).intersection(set(test_df["path"]))) == 0

os.makedirs("ml-training/data", exist_ok=True)
split_df[["path", "label", "split"]].to_csv("ml-training/data/split.csv", index=False)
sorted_classes = sorted(list(set(CLASS_MAPPING.values())))
with open("ml-training/data/class_names.json", "w") as f:
    json.dump(sorted_classes, f, indent=2)

print("Saved ml-training/data/split.csv and class_names.json successfully!")
